In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%config InlineBackend.figure_format='svg'

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import os
from pathlib import Path
from BirdDMD import dmd_loop_seqs

np.set_printoptions(suppress=True, precision=3)
# Make matplotlib use the font Andale Mono
plt.rcParams['font.family'] = 'Andale Mono'


In [ ]:

# ==========================================
# ----- Configuration -----
# ==========================================

# --- Define Parameters to Loop Through ---

# Get canonical bird names if needed, or define manually

# Or specify manually: 
bird_names = ["Toothless", "Ruby", "Drogon", "Charmander", "Rhaegal"]

perch_distances = None
# Or specify manually: perch_distances = ["5m", "7m", "9m", "12m"]

# Behaviours likely correspond to how data was segmented/saved
# Adjust this list based on the actual file prefixes you have generated
# Examples: "Initial", "TwoFlaps", "Flapping", "Gliding", "Landing"
behaviours_to_run = ["Initial"]

# Turn types (load_bird_data handles 'None' or uses it for specific filenames like 9m)
# turns_to_run = ["Straight", "Left", "Right", None] # Using None might load files without Turn specified

# --- Define Fixed DMD Parameters ---
N_MODES = 8             # Number of DMD modes to compute
D_HANKEL = 2            # Hankel delay parameter
EIG_CONSTRAINTS = {"conjugate_pairs"} # BOPDMD constraints
MIN_SEQ_LENGTH = 10 # Min frames needed (adjust if desired)
BILATERAL_STATUS = "Bilateral" # Assuming Bilateral data based on examples

# --- Define Output Directory ---
BASE_SAVE_DIR = "../data/results" # Base directory for saving results


In [ ]:

# ==========================================
# ----- Main Execution Loop -----
# ==========================================

"""Loops through specified conditions and runs DMD analysis."""
print("Starting Batch DMD Processing...")
print("==================================")
print(f"Birds: {bird_names}")
print(f"Distances: {perch_distances}")
print(f"Behaviours: {behaviours_to_run}")
print(f"Modes: {N_MODES}, Delay: {D_HANKEL}")
print(f"Base Save Dir: {BASE_SAVE_DIR}")
print("==================================")

run_counter = 0
skip_counter = 0

# Nested loops for each parameter combination
for bird in bird_names:
    for behaviour in behaviours_to_run:

        # Construct a descriptive save path for this specific combination
        # Capitalize bird name for path consistency if desired
        bird_cap = bird.capitalize()
        current_save_path = os.path.join(
            BASE_SAVE_DIR,
            f"{bird_cap}",
            f"{behaviour}",
            f"{N_MODES}Modes"
        )

        print(f"\n--- Processing: Bird={bird}, Behaviour={behaviour} ---")
        print(f"Output Path: {current_save_path}")

        # Create the directory if it doesn't exist
        if not os.path.exists(current_save_path):
            os.makedirs(current_save_path)
            print(f"Created directory: {current_save_path}")

        try:
            # Call the existing function which loads data and loops through sequences
            # Note: load_bird_data within dmd_loop_seqs needs to find the correct file
            # based on these parameters (e.g., using naming conventions)
            dmd_loop_seqs(
                bird_name=bird,
                save_path=current_save_path,
                behaviour=behaviour,
                perch_distance=perch_distances,
                bilateral=BILATERAL_STATUS,
                turn=None, # Pass the turn parameter
                n_Modes=N_MODES,
                min_seq_length=MIN_SEQ_LENGTH,
                eig_constraints=EIG_CONSTRAINTS,
                d=D_HANKEL
            )
            run_counter += 1
            print(f"--- Successfully processed combination ---")

        except FileNotFoundError as e:
            print(f"--- Skipping combination: Data file not found for these parameters. ---")
            print(f"   (Error details: {e})")
            # Optionally remove the created directory if no data was found
            # try:
            #     os.rmdir(current_save_path) # Only removes if empty
            # except OSError:
            #     pass # Directory might not be empty if some files were created before error
            skip_counter += 1
        except Exception as e:
            # Kill the loop
            print(f"--- Skipping combination due to an unexpected error: {type(e).__name__} ---")
            print(f"   (Error details: {e})")
            skip_counter += 1


print("\n==================================")
print("Batch Processing Complete.")
print(f"Successfully processed parameter combinations: {run_counter}")
print(f"Skipped parameter combinations (file not found or error): {skip_counter}")
print("==================================")


In [ ]:
def collect_all_dmd_results(base_dir, n_modes):
    """
    Load DMD results from data/results/{bird_name}/{behavior}/{n_modes}Modes/...
    with files like '05_09_017_1_dmd_results.npz' where:
    - 05 is bird ID
    - 09 is perch distance (9m)
    - 017 is sequence number
    - 1 is sub-sequence
    """
    print(f"Collecting all DMD results from: {base_dir}")
    
    # Dictionary to map bird IDs to names (based on your file naming)
    bird_id_map = {
        "01": "drogon",
        "02": "rhaegal",
        "03": "ruby",
        "04": "toothless",
        "05": "charmander"
    }
    
    # List to store each sequence's results and metadata
    all_results = []
    
    # Look through each bird directory
    for bird_dir in Path(base_dir).iterdir():
        if not bird_dir.is_dir():
            continue
            
        bird_name = bird_dir.name.lower()
        if bird_name not in [b.lower() for b in bird_names]:
            print(f"Skipping unknown bird directory: {bird_name}")
            continue
            
        print(f"\nProcessing bird: {bird_name}")
        
        # For each behavior directory
        for behav_dir in bird_dir.iterdir():
            if not behav_dir.is_dir():
                continue
                
            behavior = behav_dir.name  # e.g., "Initial"
            
            # Check for the modes directory
            modes_dir = behav_dir / f"{n_modes}Modes"
            if not modes_dir.exists() or not modes_dir.is_dir():
                print(f"  Skipping (no {n_modes}Modes dir): {behav_dir}")
                continue
            
            print(f"  Loading: Behavior={behavior}")
            
            try:
                # Load the original sequence list
                wingbeat_df, _ = runDMD.load_bird_data(
                    bird_name=bird_name,
                    behaviour=behavior,
                    bilateral="Bilateral"
                )
                
                # Load DMD results
                dmd_results_df = runDMD.load_dmd_results(wingbeat_df, str(modes_dir))
                
                if dmd_results_df is None or dmd_results_df.empty:
                    print(f"    No results found in {modes_dir}")
                    continue
                
                # Add metadata from sequence IDs
                dmd_results_df['bird'] = bird_name
                dmd_results_df['behavior'] = behavior
                
                # Extract perch distance from seqID
                def extract_distance(seqid):
                    try:
                        # Get the second part of the seqID (e.g., '09' from '05_09_017_1')
                        dist_code = seqid.split('_')[1]
                        return f"{dist_code}m"  # Convert '09' to '9m'
                    except:
                        return None
                
                dmd_results_df['distance'] = dmd_results_df['seqID'].apply(extract_distance)
                
                print(f"    Loaded {len(dmd_results_df)} sequences")
                
                # Add to master list
                all_results.append(dmd_results_df)
                
            except Exception as e:
                print(f"    Error loading {modes_dir}: {type(e).__name__} - {e}")
    
    # Combine all results
    if not all_results:
        print("\nNo results were found.")
        return None
    
    all_results_df = pd.concat(all_results, ignore_index=True)
    print(f"\nSuccessfully collected {len(all_results_df)} DMD result sequences")
    
    # Print some summary statistics
    print("\nSummary of loaded data:")
    print("Birds:", all_results_df['bird'].unique())
    print("Behaviors:", all_results_df['behavior'].unique())
    print("Distances:", all_results_df['distance'].unique())
    print("Total sequences:", len(all_results_df))
    
    return all_results_df

# Configuration
import pandas as pd
bird_names = ["Toothless", "Ruby", "Drogon", "Charmander", "Rhaegal"]
N_MODES = 8

# Set up the path and run
project_root = Path.cwd().parent  # Since we're in examples/
BASE_RESULTS_DIR = project_root / 'data' / 'results'
print(f"Loading from: {BASE_RESULTS_DIR}")

all_results_df = collect_all_dmd_results(BASE_RESULTS_DIR, N_MODES)

In [ ]:
pwd

In [ ]:
project_root = Path.cwd().parent  # Assuming notebook is in examples/ directory
data_dir = project_root / 'data'

print(data_dir)

